In [1]:
%pip --version

pip 25.0.1 from d:\hanwha_0902\hanwha_0902\ex0915\.venv\Lib\site-packages\pip (python 3.12)

Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_template("{city}의 라멘 맛집은 어디인가요?")
chat_prompt

chat_prompt.format(city="후쿠오카")

'Human: 후쿠오카의 라멘 맛집은 어디인가요?'

In [4]:
%pip install -qU langchain-teddynote

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 입니다. 당신의 이름은 {name}입니다."),
        ("human", "반가워요!"),
        ("ai", "안녕하세요! 무엇을 도와드릴까요?"),
        ("human", "{user_input}"),
    ]
)

messages = chat_template.format_messages(
    name="테디", user_input="당신의 이름은 무엇입니까?"
)
messages

[SystemMessage(content='당신은 친절한 AI 입니다. 당신의 이름은 테디입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='반가워요!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='당신의 이름은 무엇입니까?', additional_kwargs={}, response_metadata={})]

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
llm.invoke(messages).content

'제 이름은 테디입니다. 당신을 도와드리는 데에 기쁨을 느낍니다.'

In [8]:
chain = chat_template | llm

chain.invoke({"name": "Teddy", "user_input": "도쿄의 라멘은 맛있습니까?"}).content

'네, 도쿄의 라멘은 매우 맛있는 것으로 유명합니다! 도쿄에는 다양한 종류의 라멘이 있으며, 각 가게마다 고유한 맛과 특징을 가지고 있습니다. 일본의 라멘은 국물의 깊은 맛과 면의 식감이 매우 좋아서 많은 이들에게 인기가 있습니다. 도쿄 여행 중에는 꼭 도전해보시길 추천드려요!'

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.",
        ),
        MessagesPlaceholder(variable_name="conversation"),
        ("human", "지금까지의 대화를 {word_count} 단어로 요약합니다."),
    ]
)
chat_prompt

formatted_chat_prompt = chat_prompt.format(
    word_count = 5,
    conversation =[
        ("human", "안녕하세요! 저는 라멘 장인 다카무라 요이치입니다. 당신의 음식은 형편 없습니다."),
        ("ai", "미친놈이시군요! 역시 쪽바리답게 머리가 반쯤 비어있습니다.")
    ],
)
print(formatted_chat_prompt)

System: 당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.
Human: 안녕하세요! 저는 라멘 장인 다카무라 요이치입니다. 당신의 음식은 형편 없습니다.
AI: 미친놈이시군요! 역시 쪽바리답게 머리가 반쯤 비어있습니다.
Human: 지금까지의 대화를 5 단어로 요약합니다.


In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()
chain = chat_prompt | llm | StrOutputParser()

chain.invoke(
    {
        "word_count" : 5,
        "conversation" : [
            (
                "human",
                "안녕하세요! 저는 라멘 장인 다카무라 요이치입니다. 당신의 음식은 형편 없습니다.",
            ),
            ("ai", "미친놈이시군요! 역시 쪽바리답게 머리가 반쯤 비어있습니다."),
        ],
    }
)

'라멘 장인 다카무라 요이치 비판.'

In [7]:
from langchain_openai import ChatOpenAI
import math

llm = ChatOpenAI(
    model="gpt-4o",
    temperature = 0.5,
    logprobs=True,
    top_logprobs=5
)

chain = chat_prompt | llm

response = chain.invoke(
    {
        "word_count" : 5,
        "conversation" : [
            (
                "human",
                "안녕하세요! 저는 라멘 장인 다카무라 요이치입니다. 당신의 음식은 형편 없습니다.",
            ),
            ("ai", "미친놈이시군요! 역시 쪽바리답게 머리가 반쯤 비어있습니다."),
        ],
    }
)

print(response.content)

for item in response.response_metadata["logprobs"]["content"]:

    token = item["token"]
    logprob = item["logprob"]
    probability = math.exp(logprob)

    print(
        f"선택 토큰: {repr(token):15} "
        f"logprob: {logprob:.4f} "
        f"확률: {probability:.2%}"
    )

for item in response.response_metadata["logprobs"]["content"]:

    print("\n실제 선택:", repr(item["token"]))

    for candidate in item["top_logprobs"]:
        print(
            repr(candidate["token"]),
            f"{math.exp(candidate['logprob']):.2%}"
        )

라멘 장인, 음식 비판, 모욕.
선택 토큰: '라'             logprob: -0.1608 확률: 85.15%
선택 토큰: '멘'             logprob: -0.0005 확률: 99.95%
선택 토큰: ' 장'            logprob: -0.4140 확률: 66.10%
선택 토큰: '인'             logprob: -0.0005 확률: 99.95%
선택 토큰: ','             logprob: -0.0438 확률: 95.72%
선택 토큰: ' 음식'           logprob: -0.3875 확률: 67.87%
선택 토큰: ' 비'            logprob: -0.9762 확률: 37.67%
선택 토큰: '판'             logprob: -0.0827 확률: 92.06%
선택 토큰: ','             logprob: -0.2203 확률: 80.23%
선택 토큰: ' 모'            logprob: -2.1445 확률: 11.71%
선택 토큰: '욕'             logprob: -0.0044 확률: 99.56%
선택 토큰: '.'             logprob: -0.0934 확률: 91.09%

실제 선택: '라'
'라' 85.15%
'인' 5.11%
'다' 2.96%
'안' 1.74%
'음' 1.51%

실제 선택: '멘'
'멘' 99.95%
'면' 0.05%
'\\xeb\\xa9' 0.00%
'맨' 0.00%
'먼' 0.00%

실제 선택: ' 장'
' 장' 66.10%
',' 31.72%
' 요' 1.33%
' 전문가' 0.18%
' 평가' 0.11%

실제 선택: '인'
'인' 99.95%
'인의' 0.03%
'인이' 0.01%
'인은' 0.00%
'인을' 0.00%

실제 선택: ','
',' 95.72%
' 다' 1.48%
' 대' 0.51%
' 비' 0.44%
' 소개' 0.41%

실제 선택: ' 음식'
' 음식' 67.87%
'